---
title: "2.18. Tabele opisowe z gtsummary: tbl_summary"
subtitle: Charakterystyka danych i porównanie grup
---

# 2.18. Tabele opisowe z gtsummary: tbl_summary

W poprzednim podrozdziale samodzielnie obliczaliśmy statystyki, a następnie prezentowaliśmy je przez `kable()`. Teraz przygotujemy tabelę, która jednocześnie podsumowuje kilka cech: wiek, wynik ćwiczenia, kierunek studiów i udział w konsultacjach. Każda cecha wymaga właściwego sposobu opisu.

Punktem wyjścia jest [samouczek `tbl_summary()` autorów gtsummary](https://www.danieldsjoberg.com/gtsummary/articles/tbl_summary.html). Pakiet automatyzuje budowanie tabel opisowych, pozwala rozdzielić wyniki według grup oraz dodawać kolejne informacje. Przykłady poniżej wykorzystują własny, fikcyjny zbiór studentów. Gotowa tabela pomaga odczytać wyniki, ale dobór zmiennych i interpretacja pozostają zadaniem autora analizy.

**Cel:** samodzielnie przygotować tabelę charakterystyki zbioru, sprawdzić mianowniki procentów, pokazać braki oraz opisać wyniki porównania grup. Podrozdział uruchamia się niezależnie od poprzednich notebooków. Wymagany jest R co najmniej 4.1 oraz pakiety `gtsummary`, `gt` i `broom` (wykorzystywany przy testach).

## Przygotowanie środowiska

W razie potrzeby wykonaj jednorazowo w konsoli `install.packages(c("gtsummary", "gt", "broom"))`. W R Markdown i Quarto zwykle wystarcza pozostawienie obiektu `gtsummary` jako ostatniego wyrażenia komórki. W notebooku użyjemy pomocniczej funkcji wyświetlającej HTML z osadzonymi stylami. Zapisane wyniki są widoczne w e-booku bez ponownego uruchamiania R.

In [1]:
library(gtsummary)
library(gt)

pokaz_podsumowanie <- function(tabela) {
  tabela_gt <- if (inherits(tabela, "gtsummary")) gtsummary::as_gt(tabela) else tabela
  html <- gt::as_raw_html(tabela_gt, inline_css = TRUE)
  if (requireNamespace("IRdisplay", quietly = TRUE) &&
      isTRUE(getOption("jupyter.in_kernel"))) {
    IRdisplay::display_html(html)
  } else {
    cat(html, "\n")
  }
  invisible(tabela)
}

**Zadanie G.1.** Sprawdź wersje przez `packageVersion("gtsummary")` i `packageVersion("gt")`. Wyjaśnij, na którym etapie pracy powstają statystyki, a na którym ich wygląd.

## Dane: jeden student w jednym wierszu

Dwie grupy ćwiczeniowe liczą po osiem osób. Punkty mają skalę od 0 do 100. Dwa wyniki punktowe oraz jeden kierunek studiów są nieznane. `id` jest identyfikatorem, więc nie będziemy go podsumowywać jako pomiaru.

In [2]:
studenci <- data.frame(
  id = 1:16,
  grupa = factor(rep(c("A", "B"), each = 8)),
  wiek = c(19, 20, 21, 20, 22, 21, 23, 20, 20, 21, 22, 19, 24, 23, 21, 22),
  punkty = c(52, 60, 65, 70, 74, 80, 85, NA, 58, 68, 72, 77, 81, 86, 92, NA),
  kierunek = factor(c("Ekonomia", "Ekonomia", "Zarządzanie", "Ekonomia",
                      "Zarządzanie", "Ekonomia", "Zarządzanie", NA,
                      "Ekonomia", "Zarządzanie", "Zarządzanie", "Ekonomia",
                      "Zarządzanie", "Ekonomia", "Zarządzanie", "Zarządzanie")),
  konsultacje = c(TRUE, FALSE, TRUE, FALSE, TRUE, FALSE, TRUE, FALSE,
                  TRUE, TRUE, FALSE, TRUE, TRUE, FALSE, TRUE, FALSE)
)
str(studenci)
colSums(is.na(studenci))

'data.frame':	16 obs. of  6 variables:
 $ id         : int  1 2 3 4 5 6 7 8 9 10 ...
 $ grupa      : Factor w/ 2 levels "A","B": 1 1 1 1 1 1 1 1 2 2 ...
 $ wiek       : num  19 20 21 20 22 21 23 20 20 21 ...
 $ punkty     : num  52 60 65 70 74 80 85 NA 58 68 ...
 $ kierunek   : Factor w/ 2 levels "Ekonomia","Zarządzanie": 1 1 2 1 2 1 2 NA 1 2 ...
 $ konsultacje: logi  TRUE FALSE TRUE FALSE TRUE FALSE ...
         id       grupa        wiek      punkty    kierunek konsultacje 
          0           0           0           2           1           0 


Typ R i sposób podsumowania to różne pojęcia. Wiek jest liczbą, ale w małym zbiorze ma niewiele różnych wartości. Nie chcemy, by każda wartość wieku otrzymała osobny wiersz częstości. Dlatego jawnie określimy sposób jego prezentacji.

**Zadanie G.2.** Sprawdź liczebności przez `table(studenci$grupa)`. Oblicz liczbę różnych wartości wieku i liczbę dostępnych wyników punktowych. Dlaczego podsumowanie `id` średnią nie pomaga opisać studentów?

## Pierwsze podsumowanie: include, type i label

Wybierz zmienne przez `include`, a ich czytelne nazwy przez `label`. `type` steruje sposobem podsumowania: `continuous` daje jeden wiersz miar liczbowych, `categorical` wiersze kategorii, a `dichotomous` jeden wiersz wybranego zdarzenia. Dla wartości logicznych tym zdarzeniem jest domyślnie `TRUE`. Automatyczne rozpoznawanie typu warto zawsze skontrolować. Szczegóły opisuje [dokumentacja `tbl_summary()`](https://www.danieldsjoberg.com/gtsummary/reference/tbl_summary.html).

In [3]:
tabela_start <- studenci |>
  tbl_summary(
    include = c(wiek, punkty, kierunek, konsultacje),
    type = list(c(wiek, punkty) ~ "continuous", konsultacje ~ "dichotomous"),
    label = list(
      wiek ~ "Wiek [lata]", punkty ~ "Wynik ćwiczenia [pkt]",
      kierunek ~ "Kierunek studiów", konsultacje ~ "Udział w konsultacjach: tak"
    ),
    missing = "ifany",
    missing_text = "Brak danych"
  ) |>
  modify_header(label ~ "**Cecha**") |>
  modify_caption("**Charakterystyka studentów — dane fikcyjne**")
pokaz_podsumowanie(tabela_start)

"1 Median (Q1, Q3); n (%)"


Domyślny zapis cechy ciągłej przedstawia medianę oraz kwartyle Q1 i Q3; kategorii — liczebność i procent. Odczytuj przypis, bo zapis w nawiasie nie zawsze oznacza tę samą statystykę. Dwa krańce Q1 i Q3 nie są pojedynczą wartością rozstępu międzykwartylowego: IQR = Q3 − Q1.

**Zadanie G.3.** Odczytaj medianę punktów oraz liczbę osób korzystających z konsultacji. Następnie ustaw `konsultacje ~ "categorical"` i porównaj liczbę wierszy. Czy zmieniły się dane, czy tylko sposób ich prezentacji?

## Własny zestaw statystyk

Średnia wykorzystuje wszystkie wartości i jest wrażliwa na obserwacje skrajne; mediana opisuje położenie środkowe po uporządkowaniu danych. SD opisuje rozproszenie wokół średniej w jednostkach zmiennej. Wybierz miary pasujące do pytania i rozkładu danych.

W poniższej tabeli średnią i SD pokażemy z jednym miejscem po przecinku. Dla kategorii uwidocznimy także mianownik. Zapis `zmienna ~ "szablon"` wiąże zmienną z ustawieniem — nie jest tu formułą modelu regresji.

In [4]:
tabela_grupy <- studenci |>
  tbl_summary(
    by = grupa,
    include = c(wiek, punkty, kierunek, konsultacje),
    type = list(c(wiek, punkty) ~ "continuous", konsultacje ~ "dichotomous"),
    label = list(
      wiek ~ "Wiek [lata]", punkty ~ "Wynik ćwiczenia [pkt]",
      kierunek ~ "Kierunek studiów", konsultacje ~ "Udział w konsultacjach: tak"
    ),
    statistic = list(
      all_continuous() ~ "{mean} ({sd})",
      all_categorical() ~ "{n} / {N} ({p}%)"
    ),
    digits = all_continuous() ~ 1,
    percent = "column",
    missing = "ifany", missing_text = "Brak danych"
  ) |>
  modify_header(label ~ "**Cecha**") |>
  modify_caption("**Opis grup A i B: średnia (SD) oraz liczebności**")
pokaz_podsumowanie(tabela_grupy)

1 Mean (SD); n / N (%)


Selektory `all_continuous()` i `all_categorical()` wybierają cechy według sposobu podsumowania. Zaokrąglenia dotyczą prezentacji; źródłowa ramka pozostaje bez zmian.

**Zadanie G.4.** Sprawdź średnie punktów przez `tapply(studenci$punkty, studenci$grupa, mean, na.rm = TRUE)`. Następnie zmień szablon punktów na `"{median} ({p25}, {p75})"`. Wyjaśnij różnicę między obiema wersjami, patrząc również na przypis.

## Braki danych i mianowniki procentów

Osiem osób w nagłówku grupy A nie oznacza ośmiu znanych wartości każdej cechy. Średnia punktów wykorzystuje siedem wyników. Kierunek studiów jest znany również dla siedmiu osób: cztery studiują ekonomię, czyli 4/7, około 57%. Nie jest to 4/8 = 50%.

`missing = "ifany"` pokazuje wiersz braku, gdy występuje przynajmniej jeden brak. `"always"` pokazuje go również przy zerowej liczbie braków, a `"no"` ukrywa wiersz; **nie uzupełnia brakujących danych**. Brak w zmiennej grupującej wymaga osobnej kontroli, ponieważ obserwacje bez ustalonej grupy nie tworzą zwykłej kolumny porównania.

`percent = "column"` odpowiada na pytanie „jaki odsetek osób o znanej wartości w danej grupie należy do kategorii?”. `percent = "row"` odpowiada na inne pytanie: „jak osoby z danej kategorii dzielą się między grupy?”.

In [5]:
tabela_procenty <- studenci |>
  tbl_summary(
    by = grupa, include = kierunek,
    label = kierunek ~ "Kierunek studiów",
    statistic = all_categorical() ~ "{n} / {N} ({p}%)",
    percent = "row", missing = "ifany", missing_text = "Brak danych"
  ) |>
  modify_header(label ~ "**Cecha**") |>
  modify_caption("**Rozkład kierunku między grupami — procenty wierszowe**")
pokaz_podsumowanie(tabela_procenty)

1 n / N (%)


**Zadanie G.5.** Dla ekonomii wyjaśnij różnicę między 4/7 w grupie A przy procentach kolumnowych i 4/7 przy procentach wierszowych. Liczby przypadkiem są takie same, ale jakie osoby tworzą każdy z mianowników? Następnie porównaj mianowniki dla zarządzania. Zmień `missing` na `"no"` i sprawdź, czy zniknęły braki w danych, czy tylko ich wiersz.

## Ogółem i liczba dostępnych obserwacji

Dodajmy charakterystykę całego zbioru przez `add_overall()` oraz liczbę znanych wartości każdej cechy przez `add_n()`. W kolumnie „Dostępne” punktów będzie 14, kierunku 15, a wieku 16. To informacja o danych, na których oparto podsumowanie, a nie nowa grupa studentów.

In [6]:
tabela_pelna <- tabela_grupy |>
  add_overall(last = TRUE, col_label = "**Ogółem**, N = {N}") |>
  add_n(statistic = "{N_nonmiss}", col_label = "**Dostępne**") |>
  modify_spanning_header(c("stat_1", "stat_2") ~ "**Grupa ćwiczeniowa**") |>
  bold_labels()
pokaz_podsumowanie(tabela_pelna)

1 Mean (SD); n / N (%)


Statystyka ogółem jest liczona z obserwacji. Nie obliczaj jej automatycznie jako nieważonej średniej średnich grup: przy różnych liczbach znanych wyników potrzebne są odpowiednie wagi.

**Zadanie G.6.** Zamień jeden znany wynik grupy A na `NA` w kopii danych i zbuduj tabelę ponownie. Przewidź, które liczebności się zmienią. Porównaj średnią ogólną z prostą średnią dwóch średnich grupowych.

## Kilka miar jednej zmiennej: continuous2

Czasem czytelnik potrzebuje równocześnie miary położenia i zakresu. Typ `continuous2` układa kolejne szablony w osobnych wierszach. Taka tabela jest dłuższa, ale nie wymaga upychania wszystkich miar w jednej komórce.

In [7]:
tabela_punkty <- studenci |>
  tbl_summary(
    by = grupa, include = punkty,
    type = punkty ~ "continuous2",
    label = punkty ~ "Wynik ćwiczenia [pkt]",
    statistic = punkty ~ c("{mean} ({sd})", "{median} ({p25}, {p75})", "{min}; {max}"),
    digits = punkty ~ 1,
    missing = "always", missing_text = "Brak danych"
  ) |>
  modify_header(label ~ "**Miara**") |>
  modify_caption("**Punkty — różne sposoby opisu rozkładu**")
pokaz_podsumowanie(tabela_punkty)

Tabela opisowa HTML — wynik kodu powyżej.

Przy ręcznej kontroli kwartylów sprawdź algorytm: `{p25}` i `{p75}` w `tbl_summary()` korzystają z kwantyli typu 2, podczas gdy bazowe `quantile()` domyślnie używa typu 7. Wyniki dla małych prób mogą się różnić. Porównuj z `quantile(x, c(.25, .75), na.rm = TRUE, type = 2)`.

**Zadanie G.7.** Odtwórz medianę i kwartyle grupy A w bazowym R. Dopisz w kopii danych wynik 100 zamiast 52 i porównaj zmianę średniej oraz mediany. Czy oba opisy reagują tak samo?

## Wprowadzenie do add_p: wynik testu wymaga interpretacji

Do tabeli opisowej można dołączyć wartości p. Najpierw ustal pytanie, rodzaj danych i schemat badania. Tutaj traktujemy dwie grupy jako niezależne i pokazujemy jedynie mechanikę testów na danych fikcyjnych. Dla punktów wybierzemy test sumy rang Wilcoxona, a dla kierunku i konsultacji test dokładny Fishera. Ten drugi nadaje się do małych tabel liczebności; testowanie nie jest konieczną częścią każdej tabeli opisowej.

In [8]:
tabela_testy <- tabela_grupy |>
  add_p(
    include = c(punkty, kierunek, konsultacje),
    test = list(punkty ~ "wilcox.test", c(kierunek, konsultacje) ~ "fisher.test"),
    test.args = punkty ~ list(exact = FALSE),
    pvalue_fun = label_style_pvalue(digits = 2)
  ) |>
  modify_header(p.value ~ "**Wartość p**") |>
  modify_caption("**Porównanie grup — demonstracja testów na danych fikcyjnych**")
pokaz_podsumowanie(tabela_testy)

1 Mean (SD); n / N (%)
2 Wilcoxon rank sum test; Fisher’s exact test


`exact = FALSE` wybiera przybliżenie asymptotyczne testu Wilcoxona; domyślna korekta ciągłości pozostaje włączona. Wiek opisujemy bez wykonywania testu, dlatego jego pole wartości p jest puste. Dobór i opcje testów opisuje [dokumentacja `add_p()`](https://www.danieldsjoberg.com/gtsummary/reference/add_p.tbl_summary.html).

Wartość p dotyczy zgodności danych z hipotezą zerową i założeniami testu; nie jest prawdopodobieństwem, że hipoteza zerowa jest prawdziwa. Małe p nie mierzy wielkości różnicy, a duże p nie dowodzi równości grup. Test Wilcoxona nie jest ogólnie testem samych median: interpretacja jako przesunięcia położenia wymaga dodatkowych założeń o rozkładach. Niezależność obserwacji wynika ze sposobu zbierania danych, a nie z tabeli. Przy wielu testach trzeba uwzględnić problem wielokrotnych porównań; funkcja `add_q()` służy do dodawania skorygowanych wartości.

**Zadanie G.8.** Sprawdź wynik punktów przez `wilcox.test(punkty ~ grupa, data = studenci, exact = FALSE)`. Odczytaj nazwy testów w przypisie tabeli. Napisz dwa zdania: jedno o opisowej różnicy wyników, drugie o tym, czego sama wartość p nie pozwala stwierdzić. Nie formułuj wniosku o przyczynowym wpływie grupy.

## Ostatnie poprawki wyglądu i przejście do gt

W obiekcie `gtsummary` najpierw dopracuj treść i nagłówki. Na końcu `as_gt()` otwiera możliwość formatowania przez pakiet `gt`. Po konwersji pracujesz już z obiektem innej klasy — zachowaj wcześniejszą tabelę, jeżeli chcesz później użyć np. `add_n()`.

In [9]:
tabela_do_raportu <- tabela_pelna |>
  as_gt() |>
  gt::tab_source_note(
    source_note = "Źródło: własne dane fikcyjne. SD: odchylenie standardowe; braki nie są zerami."
  ) |>
  gt::tab_options(table.font.size = gt::px(14))
pokaz_podsumowanie(tabela_do_raportu)

1 Mean (SD); n / N (%)
Źródło: własne dane fikcyjne. SD: odchylenie standardowe; braki nie są zerami.


Funkcja pomocnicza używa `gt::as_raw_html(inline_css = TRUE)`, dzięki czemu styl jest osadzony przy elementach tabeli. Opis tego mechanizmu znajduje się w [dokumentacji gt](https://gt.rstudio.com/reference/as_raw_html.html). W notebooku mamy zarówno kod, jak i gotowy wynik.

**Zadanie G.9.** Zmień podpis przez `modify_caption()` przed konwersją, a po konwersji dodaj własną notatkę źródłową. Sprawdź `class(tabela_pelna)` i `class(tabela_do_raportu)`.

## Eksport oraz związek z kableExtra

Samodzielny plik HTML zapisz w katalogu własnego projektu następującym poleceniem w konsoli:

```text
gt::gtsave(tabela_do_raportu, filename = "charakterystyka_studentow.html", inline_css = TRUE)
```

Otwórz go w przeglądarce i sprawdź podpisy, przypisy oraz szerokość tabeli. Jeśli chcesz kontynuować formatowanie narzędziami poznanymi wcześniej, pakiet udostępnia również `gtsummary::as_kable_extra(tabela_pelna, format = "html")` (wymaga `kableExtra`). To alternatywna konwersja końcowa. Nie każda cecha wyglądu przenosi się identycznie między silnikami, dlatego wynik eksportu trzeba obejrzeć.

**Zadanie G.10.** Zapisz tabelę jako HTML. Opcjonalnie skonwertuj `tabela_pelna` przez `as_kable_extra()` i wyświetl ją sposobem z podrozdziału 2.17. Porównaj treść, nagłówki oraz przypisy obu wersji.

## Mały projekt: charakterystyka grup studentów

**Zadanie G.11.** Utwórz własny fikcyjny zbiór co najmniej 20 osób: dwie grupy, cecha liczbowa, kategoria z trzema poziomami i cecha logiczna. Wprowadź dwa jawnie oznaczone braki.

1. Zbuduj `tbl_summary()` z opisowymi etykietami i jednostkami, bez identyfikatora.
2. Wymuś odpowiednie typy podsumowań i pokaż braki.
3. Dodaj kolumnę ogółem oraz liczbę dostępnych obserwacji.
4. Sprawdź ręcznie jeden procent i jedną średnią; zapisz ich mianowniki.
5. Zapisz tabelę HTML i dopisz trzy zdania opisu, odnosząc się do miar oraz braków.
6. Opcjonalnie sformułuj jedno pytanie porównawcze, uzasadnij wybór testu i dodaj jego wynik. Zaznacz ograniczenia małego, fikcyjnego zbioru.

**Punkty kontrolne dla przykładu w podrozdziale:** 16 osób; po 8 w grupie; 14 znanych wyników punktowych; 15 znanych kierunków; 9 osób korzystających z konsultacji. Średnia punktów wynosi około 69,43 w A i 76,29 w B. Procenty kierunków w każdej kolumnie tabeli grupowej odnoszą się do dostępnych wartości tej zmiennej.

## Sprawdź rozumienie

**Zadanie G.12.** Wyjaśnij własnymi słowami: dlaczego `missing = "no"` nie rozwiązuje problemu braków? Czym różnią się procenty wierszowe i kolumnowe? Dlaczego typ liczbowy R nie wystarcza do wybrania podsumowania? Co jest potrzebne do interpretacji p poza samą liczbą? Które czynności wykonasz przed `as_gt()`, a które po nim?

Dalsze przykłady, w tym tabele krzyżowe i podsumowania danych z wagami badania, zawiera [samouczek tbl_summary](https://www.danieldsjoberg.com/gtsummary/articles/tbl_summary.html). W danych złożonych badań reprezentacyjnych potrzebny jest opis schematu losowania i właściwe narzędzia, np. `tbl_svysummary()`; zwykła tabela nie uwzględnia go automatycznie.

[← Wróć do formatowania tabel z kable i kableExtra](rozdzial-2-kable.ipynb)

[Przejdź do podrozdziału 2.19: zapytania tekstowe i REGEX →](rozdzial-2-regex.ipynb)